In [ ]:
import pandas as pd

df = pd.read_csv("data.csv")
df = df[['source', 'translation']].dropna()

def clean_text(text):
    text = text.lower()
    text = text.replace('"', '')
    text = text.replace('<gap>', ' <sep> ')
    return text

df['source'] = df['source'].apply(clean_text)
df['translation'] = df['translation'].apply(clean_text)

df['translation'] = "<sos> " + df['translation'] + " <eos>"

In [ ]:
def tokenize(x): return x.split()

df['src_tokens'] = df['source'].apply(tokenize)
df['tgt_tokens'] = df['translation'].apply(tokenize)

from collections import Counter

def build_vocab(tokens):
    vocab = {"<pad>":0, "<unk>":1}
    counter = Counter()

    for t in tokens:
        counter.update(t)

    for word, freq in counter.items():
        if freq >= 2:
            vocab[word] = len(vocab)

    return vocab

src_vocab = build_vocab(df['src_tokens'])
tgt_vocab = build_vocab(df['tgt_tokens'])

def encode(tokens, vocab):
    return [vocab.get(t, vocab["<unk>"]) for t in tokens]

df['src_ids'] = df['src_tokens'].apply(lambda x: encode(x, src_vocab))
df['tgt_ids'] = df['tgt_tokens'].apply(lambda x: encode(x, tgt_vocab))

In [ ]:
import torch
from torch.utils.data import Dataset, DataLoader
from torch.nn.utils.rnn import pad_sequence

class TranslationDataset(Dataset):
    def __init__(self, df):
        self.src = df['src_ids'].tolist()
        self.tgt = df['tgt_ids'].tolist()

    def __len__(self):
        return len(self.src)

    def __getitem__(self, idx):
        return torch.tensor(self.src[idx]), torch.tensor(self.tgt[idx])

def collate_fn(batch):
    src, tgt = zip(*batch)
    src = pad_sequence(src, padding_value=0)
    tgt = pad_sequence(tgt, padding_value=0)
    return src, tgt

dataloader = DataLoader(TranslationDataset(df), batch_size=32, shuffle=True, collate_fn=collate_fn)

In [ ]:
import torch.nn as nn

class Attention(nn.Module):
    def __init__(self, enc_hid_dim, dec_hid_dim):
        super().__init__()

        self.attn = nn.Linear(enc_hid_dim * 2 + dec_hid_dim, dec_hid_dim)
        self.v = nn.Linear(dec_hid_dim, 1, bias=False)

    def forward(self, hidden, encoder_outputs):
        src_len = encoder_outputs.shape[0]

        hidden = hidden.repeat(src_len, 1, 1)

        energy = torch.tanh(self.attn(
            torch.cat((hidden, encoder_outputs), dim=2)
        ))

        attention = self.v(energy).squeeze(2)

        return torch.softmax(attention, dim=0)

In [ ]:
class Encoder(nn.Module):
    def __init__(self, input_dim, emb_dim, hid_dim):
        super().__init__()

        self.embedding = nn.Embedding(input_dim, emb_dim)
        self.lstm = nn.LSTM(emb_dim, hid_dim, bidirectional=True)

    def forward(self, src):
        embedded = self.embedding(src)
        outputs, (hidden, cell) = self.lstm(embedded)

        return outputs, hidden, cell

In [ ]:
class Decoder(nn.Module):
    def __init__(self, output_dim, emb_dim, enc_hid_dim, dec_hid_dim, attention):
        super().__init__()

        self.embedding = nn.Embedding(output_dim, emb_dim)
        self.attention = attention

        self.lstm = nn.LSTM(enc_hid_dim*2 + emb_dim, dec_hid_dim)

        self.fc = nn.Linear(enc_hid_dim*2 + dec_hid_dim + emb_dim, output_dim)

    def forward(self, input, hidden, cell, encoder_outputs):
        input = input.unsqueeze(0)
        embedded = self.embedding(input)

        a = self.attention(hidden[-1], encoder_outputs)
        a = a.unsqueeze(1).permute(1, 2, 0)

        encoder_outputs = encoder_outputs.permute(1, 0, 2)

        weighted = torch.bmm(a, encoder_outputs)
        weighted = weighted.permute(1, 0, 2)

        lstm_input = torch.cat((embedded, weighted), dim=2)

        output, (hidden, cell) = self.lstm(lstm_input, (hidden, cell))

        prediction = self.fc(torch.cat((output, weighted, embedded), dim=2).squeeze(0))

        return prediction, hidden, cell

In [ ]:
class Seq2Seq(nn.Module):
    def __init__(self, encoder, decoder, device):
        super().__init__()
        self.encoder = encoder
        self.decoder = decoder
        self.device = device

    def forward(self, src, tgt, teacher_forcing_ratio=0.5):
        batch_size = tgt.shape[1]
        tgt_len = tgt.shape[0]
        vocab_size = self.decoder.fc.out_features

        outputs = torch.zeros(tgt_len, batch_size, vocab_size).to(self.device)

        encoder_outputs, hidden, cell = self.encoder(src)

        # fix bidirectional hidden
        hidden = torch.cat((hidden[-2], hidden[-1]), dim=1).unsqueeze(0)
        cell = torch.cat((cell[-2], cell[-1]), dim=1).unsqueeze(0)

        input = tgt[0]

        for t in range(1, tgt_len):
            output, hidden, cell = self.decoder(input, hidden, cell, encoder_outputs)
            outputs[t] = output

            top1 = output.argmax(1)
            input = tgt[t] if torch.rand(1).item() < teacher_forcing_ratio else top1

        return outputs

In [ ]:
device = torch.device("cuda" if torch.cuda.is_available() else "cpu")

attn = Attention(256, 256)

encoder = Encoder(len(src_vocab), 128, 256)
decoder = Decoder(len(tgt_vocab), 128, 256, 256, attn)

model = Seq2Seq(encoder, decoder, device).to(device)

import torch.optim as optim
optimizer = optim.Adam(model.parameters())

criterion = nn.CrossEntropyLoss(ignore_index=0)

In [ ]:
from tqdm import tqdm

for epoch in range(10):
    model.train()
    total_loss = 0

    progress_bar = tqdm(dataloader, desc=f"Epoch {epoch}")

    for src, tgt in progress_bar:
        src, tgt = src.to(device), tgt.to(device)

        optimizer.zero_grad()

        output = model(src, tgt)

        output_dim = output.shape[-1]

        output = output[1:].view(-1, output_dim)
        tgt = tgt[1:].reshape(-1)

        loss = criterion(output, tgt)
        loss.backward()

        # 🔥 (important) gradient clipping
        torch.nn.utils.clip_grad_norm_(model.parameters(), 1)

        optimizer.step()

        total_loss += loss.item()

        progress_bar.set_postfix(loss=loss.item())

    avg_loss = total_loss / len(dataloader)

    print(f"\nEpoch {epoch} Summary:")
    print(f"Loss: {avg_loss:.4f}")

In [ ]:
bleu, chrf, combined = evaluate_model(
    model,
    df.sample(100),
    src_vocab,
    tgt_vocab,
    device
)

print(f"BLEU: {bleu:.2f}")
print(f"chrF++: {chrf:.2f}")
print(f"BLEU x chrF++: {combined:.2f}")